In [1]:
%pip install catboost lightgbm optuna holidays duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 21.5 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import duckdb

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
%matplotlib inline

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, TimeSeriesSplit, cross_val_predict, cross_val_score, cross_validate
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool

import optuna
from typing import Any
import warnings

import holidays

In [5]:
def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

DRIVE_FOLDER_NAME = "Store Sales"

In [6]:
import plotly.io as pio
pio.renderers.default = "vscode"

In [7]:
TARGET = "sales"
N_SPLITS = 5
SEED = 42
eps = 1e-5

def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

non_relevant_columns = ['id', 'source']


In [8]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/train.csv')
    test_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  train_df['date'] =  pd.to_datetime(train_df['date'])
  test_df['date'] =  pd.to_datetime(test_df['date'])
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

In [9]:
def load_helper_data(option: str="local"):
  '''
  returns oil_df, transactions_df, holidays_events_df, stores_df

  '''
  if option == "local":
    oil_df = pd.read_csv('oil.csv')
    transactions_df = pd.read_csv('transactions.csv')
    holidays_events_df = pd.read_csv('holidays_events.csv')
    stores_df = pd.read_csv('stores.csv')


  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    oil_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/oil.csv')
    transactions_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/transactions.csv')
    holidays_events_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/holidays_events.csv')
    stores_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/stores.csv')
    
  oil_df['date'] =  pd.to_datetime(oil_df['date'])
  transactions_df['date'] =  pd.to_datetime(transactions_df['date'])
  holidays_events_df['date'] =  pd.to_datetime(holidays_events_df['date'])
  


  return oil_df, transactions_df, holidays_events_df, stores_df

In [10]:
def data_splitter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train_dataset = df[df['source'] == 'train'].drop(columns=non_relevant_columns + additional_drop_columns)
    submission_dataset = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train_dataset.drop(columns=[TARGET])
    y_train = train_dataset[TARGET]

    return X_train, y_train, submission_dataset

In [ ]:
train_df, test_df, df = load_data(ENVIRONMENT)
oil_df, transactions_df, holidays_events_df, stores_df = load_helper_data(ENVIRONMENT)

# EDA

## Overall Info

### Main DF

In [12]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,source
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,train
1,1,2013-01-01,1,BABY CARE,0.0,0,train
2,2,2013-01-01,1,BEAUTY,0.0,0,train
3,3,2013-01-01,1,BEVERAGES,0.0,0,train
4,4,2013-01-01,1,BOOKS,0.0,0,train


In [13]:
df.groupby('source').agg(min_date=('date', 'min'), max_date=('date', 'max'))

,min_date,max_date
source,,
test,2017-08-16,2017-08-31
train,2013-01-01,2017-08-15


In [0]:
df.describe(include='all')

### Stores

In [0]:
stores_df.head()

In [0]:
stores_df.describe(include='all')

### Transactions

In [0]:
transactions_df.head()

In [0]:
transactions_df.describe(include='all')

### Holidays

In [0]:
holidays_events_df.head()

In [0]:
holidays_events_df.describe()

### Oil

In [0]:
oil_df.head()

In [0]:
oil_df.describe()

In [0]:
df.head()

In [0]:
transactions_df

# Merging datasets

In [14]:
initial_length = len(df)

In [15]:
initial_length

3029400

In [16]:
df.isna().sum()

,0
id,0
date,0
store_nbr,0
family,0
sales,28512
onpromotion,0
source,0


In [20]:
transactions_df

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922
...,...,...,...
83483,2017-08-15,50,2804
83484,2017-08-15,51,1573
83485,2017-08-15,52,2255
83486,2017-08-15,53,932


In [ ]:
temp_df = train_df.merge(transactions_df, on=['date', 'store_nbr'], how='left')

In [24]:
temp_df.loc[temp_df['transactions'].isna()].head()

,id,date,store_nbr,family,sales,onpromotion,source,transactions
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,train,NaN
1,1,2013-01-01,1,BABY CARE,0.0,0,train,NaN
2,2,2013-01-01,1,BEAUTY,0.0,0,train,NaN
3,3,2013-01-01,1,BEVERAGES,0.0,0,train,NaN
4,4,2013-01-01,1,BOOKS,0.0,0,train,NaN


In [ ]:
duckdb.sql("""--sql
    SELECT
        SUM(CASE WHEN sales = 0 THEN 1 ELSE 0 END) AS zero_sales,
        SUM(CASE WHEN sales > 0 THEN 1 ELSE 0 END) AS more_sales
    FROM temp_df
    WHERE transactions IS NULL
""").df()

,zero_sales,more_sales
0,242536.0,3248.0


In [40]:
temp_df.loc[(temp_df['sales'] > 0) & (temp_df['transactions'].isna())]

,id,date,store_nbr,family,sales,onpromotion,source,transactions
301191,301191,2013-06-19,10,AUTOMOTIVE,3.000,0,train,NaN
301194,301194,2013-06-19,10,BEVERAGES,515.000,0,train,NaN
301196,301196,2013-06-19,10,BREAD/BAKERY,94.000,0,train,NaN
301198,301198,2013-06-19,10,CLEANING,695.000,0,train,NaN
301199,301199,2013-06-19,10,DAIRY,43.000,0,train,NaN
...,...,...,...,...,...,...,...,...
2428794,2428794,2016-09-27,7,PLAYERS AND ELECTRONICS,12.000,0,train,NaN
2428795,2428795,2016-09-27,7,POULTRY,674.468,0,train,NaN
2428796,2428796,2016-09-27,7,PREPARED FOODS,172.204,0,train,NaN
2428797,2428797,2016-09-27,7,PRODUCE,5306.265,1,train,NaN


In [48]:
(temp_df.loc[temp_df['transactions'].isna(), 'date']
 .value_counts()
 .head(20)) 

,count
date,
2016-01-01,1782
2016-01-03,1782
2017-01-01,1749
2015-01-01,1749
2013-01-01,1749
2014-01-01,1716
2016-01-04,1320
2016-01-02,594
2013-06-19,363


In [0]:
holidays_events_df['locale_name'].unique()

In [0]:
display(stores_df)

In [0]:
df.sort_values(by='date', ascending=True).reset_index(drop=True).head(10)

In [0]:
df.index[df['date'] == '2017-01-01']

In [0]:
dates = np.sort(df["date"].unique())

tscv = TimeSeriesSplit(n_splits=4, test_size=16)   # 16 days = competition horizon

for i in enumerate(tscv.split(dates)):
    print(i)

In [0]:
stores_df['state'].unique()